# PAM Hotels — Agente de Trazabilidad UUID
## PoC: Reconciliación Automática FBL3N ↔ FBL1N

Este notebook implementa el proceso completo de matching de gastos del libro mayor con sus facturas CFDI y pagos.

**Fuentes de datos:**
- `AuxGastos.xlsx` — Libro Mayor (FBL3N): ~2,400 registros de gasto por mes
- `3400.xlsx` — Partidas de Proveedor (FBL1N): ~37,500 registros
- `Acreedor.xlsx` — Datos Maestros de Proveedores: ~24,600 registros

**Estrategias de matching (en orden de prioridad):**
1. `UUID_DIRECTO` — UUID del CFDI presente en ambos archivos
2. `DOCUMENTO_CRUCE` — Número de documento contable (BELNR)
3. `FUZZY_CON_ACREEDOR` — Fecha ±5d + importe ±1% + acreedor + similitud texto
4. `FUZZY_SIN_ACREEDOR` — Fecha ±3d + importe ±1% + token overlap
5. `NO_RESUELTO` — Sin match automático, requiere revisión manual

In [1]:
%pip install pandas openpyxl xlsxwriter python-dateutil tqdm -q


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import os
import warnings
from datetime import datetime, timedelta
from difflib import SequenceMatcher

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
tqdm.pandas()
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 50)
print('Imports OK')

Imports OK


/Users/tadeo/gitlab/agentPAM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ─── Paths ──────────────────────────────────────────────────────────────────
INPUT_DIR  = 'files/'
OUTPUT_DIR = 'output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUXGASTOS_FILE = os.path.join(INPUT_DIR, 'AuxGastos.xlsx')
PAGOS_DIR      = os.path.join(INPUT_DIR, 'Pagos/')   # One FBL1N file per society
ACREEDOR_FILE  = os.path.join(INPUT_DIR, 'Acreedor.xlsx')
OUTPUT_FILE    = os.path.join(OUTPUT_DIR, 'ReporteFinal_3400_202002.xlsx')

# ─── Column index maps for FBL1N full format (3400.xlsx — 123 columns) ───────
FBL1N_COLS = {
    'referencia': 0, 'acreedor_id': 3, 'nombre_proveedor': 4,
    'clave_ct': 5, 'nro_documento': 6, 'soc_gl': 7, 'clase_doc': 8,
    'fecha_documento': 10, 'fecha_contab': 11, 'fecha_pago': 13,
    'importe': 18, 'moneda': 19, 'sociedad': 20, 'cta_banco': 21,
    'texto': 22, 'indicador_ii': 67, 'via_pago': 68,
    'condicion_pago': 69, 'usuario': 70, 'uuid': 71,
    'fecha_compensacion': 72, 'doc_compensacion': 73, 'anulacion': 74,
}

# ─── Column name map for FBL1N compact format (all other societies — 23 cols) ─
FBL1N_COLS_COMPACT = {
    'Acreedor':                   'acreedor_id',
    'Referencia':                 'referencia',
    'Cta.contrapartida':          'cta_banco',
    'Descripción de proveedor':   'nombre_proveedor',
    'Clase de documento':         'clase_doc',
    'Nº documento':               'nro_documento',
    'Fecha de documento':         'fecha_documento',
    'Fe.contabilización':         'fecha_contab',
    'Fecha de pago':              'fecha_pago',
    'Importe en moneda doc.':     'importe',
    'Moneda del documento':       'moneda',
    'Sociedad':                   'sociedad',
    'Texto':                      'texto',
    'Folio Fiscal':               'uuid',
    'Fecha compensación':         'fecha_compensacion',
    'Doc.compensación':           'doc_compensacion',
    'Nombre del usuario':         'usuario',
    'Anulado con':                'anulacion',
}

# ─── AuxGastos column index map ───────────────────────────────────────────────
AUXGASTOS_COLS = {
    'sociedad': 0, 'fecha_contab': 1, 'ejercicio': 2, 'periodo': 3,
    'posicion': 4, 'nro_documento': 5, 'clase_doc': 6, 'clave_ct': 7,
    'cuenta_gl': 8, 'descripcion_cuenta': 9, 'importe': 10, 'moneda': 11,
    'importe_ml': 12, 'moneda_local': 13, 'indicador_iva': 14,
    'folio_fiscal': 15, 'uuid': 16, 'cta_contrapartida': 17,
    'acreedor_id': 18, 'nombre_proveedor': 19, 'soc_gl': 20,
    'anulacion': 21, 'fecha_registro': 22, 'referencia': 23,
    'asignacion': 24, 'texto': 25, 'centro_costo': 26,
    'usuario': 27, 'doc_compensacion': 28,
}

ACREEDOR_COLS = {
    'sociedad': 0, 'acreedor_id': 1, 'nombre_proveedor': 4,
    'rfc': 13, 'ramo': 16, 'cuenta_asociada': 18,
    'condicion_pago': 19, 'soc_gl': 22,
}

# ═══════════════════════════════════════════════════════════════════════════════
# TOLERANCIAS DE MATCHING — ajusta estos valores para afinar los resultados
# ═══════════════════════════════════════════════════════════════════════════════

# ── S3: FUZZY_CON_ACREEDOR (registros CON acreedor_id) ────────────────────────
# Busca en FBL1N filtrando primero por mismo vendor, luego por fecha y texto.
S3_FECHA_TOLERANCIA_DIAS  = 5     # ± días entre Fe.contab. de AuxGastos y FBL1N
                                  # Aumentar si pagos llegan tarde al sistema.
S3_IMPORTE_TOLERANCIA_PCT = None  # ± % sobre importe_ml de AuxGastos.
                                  # NOTA: GL registra líneas parciales (costo/centro);
                                  # FBL1N registra el total de la factura. Por eso
                                  # se recomienda un valor alto (0.50–1.00) o None
                                  # para desactivar el filtro de importe en S3.
                                  # Cambiar a None para desactivar.

# ── S4: FUZZY_SIN_ACREEDOR (registros SIN acreedor_id) ───────────────────────
# Sin ancla de vendor, filtra FBL1N solo por fecha e importe.
# Ventana más estrecha para reducir falsos positivos.
S4_FECHA_TOLERANCIA_DIAS  = 3     # ± días (más estricto que S3 porque no hay vendor)
S4_IMPORTE_TOLERANCIA_PCT = 0.01  # ±1% sobre importe_ml. Aquí los importes sí
                                  # deberían coincidir mejor (misma línea GL vs pago).

# ── Umbrales de similitud de texto (aplican a S3 y S4) ───────────────────────
TEXT_SCORE_ALTO  = 0.70  # score ≥ este valor → DIFUSO_ALTO  (confianza alta)
TEXT_SCORE_MEDIO = 0.40  # score ≥ este valor → DIFUSO_MEDIO (revisar antes de usar)
                         # score < TEXT_SCORE_MEDIO → DIFUSO_BAJO (siempre revisar)

# ── Prefijos de cuentas GL de gasto ──────────────────────────────────────────
CUENTAS_GL_GASTO_PREFIX = ['71', '72', '73', '74', '75', '76', '77']
RFC_EXTRANJERO          = 'XEXX010101000'

# ── Colores de output Excel ───────────────────────────────────────────────────
OUTPUT_COLORS = {
    'EXACTO':       'C6EFCE',
    'DIFUSO_ALTO':  'FFEB9C',
    'DIFUSO_MEDIO': 'FFCC99',
    'DIFUSO_BAJO':  'FFC7CE',
    'NO_RESUELTO':  'FF6666',
    'ANULADO':      'D9D9D9',
}

print(f'Config OK  |  Pagos: {PAGOS_DIR}  |  Output: {OUTPUT_FILE}')
# ─── Ecommerce (CFDIs Recibidos) ────────────────────────────────────────────
ECOMMERCE_FILE  = os.path.join(INPUT_DIR, 'ecommerce.xlsx')
ECOMMERCE_SHEET = 'CIK130405M86 CFULL Recibidos 20'
ECOMMERCE_COLS  = {
    'xml_filename':    0,
    'uuid':            1,   # lowercase in file — normalize via clean_uuid()
    'estatus':         2,   # 'Vigente' | 'Cancelado' — filter on load
    'fecha_cancelacion':3,
    'tipo_cfdi':       8,
    'serie':           9,
    'folio':           10,
    'fecha_emision':   11,  # Excel integer serial — use excel_serial_to_date()
    'fecha_timbrado':  21,
    'receptor_rfc':    23,
    'receptor_nombre': 24,
    'emisor_rfc':      31,  # RFC del proveedor — DATO FISCAL OBJETIVO
    'emisor_nombre':   32,  # Razón social del proveedor
    'concepto':        40,  # Para matching semántico difuso
    'subtotal':        45,
    'iva':             52,
    'total_xml':       60,
    'total_mxn':       61,  # Usar este para matching (ya convertido a MXN)
    'tipo_cambio':     62,
    'moneda':          63,
    'forma_pago':      65,
    'forma_pago_desc': 66,
    'metodo_pago':     67,
    'metodo_pago_desc':68,
}


Config OK  |  Pagos: files/Pagos/  |  Output: output/ReporteFinal_3400_202002.xlsx


In [5]:
def parse_sap_date(val):
    """Parse SAP 'DD.MM.YYYY' string or passthrough datetime → date."""
    if val is None:
        return None
    try:
        if pd.isna(val):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(val, datetime):
        return val.date()
    if isinstance(val, str):
        s = val.strip()
        if len(s) == 10:
            try:
                return datetime.strptime(s, '%d.%m.%Y').date()
            except ValueError:
                pass
    return None


def clean_uuid(val):
    """Normalize UUID: strip, uppercase; None if blank or too short."""
    try:
        if pd.isna(val):
            return None
    except (TypeError, ValueError):
        pass
    if val is None:
        return None
    s = str(val).strip().upper()
    return s if len(s) >= 32 else None


def text_score(a, b):
    """SequenceMatcher ratio (case-insensitive)."""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, str(a).lower().strip(), str(b).lower().strip()).ratio()


def token_overlap(a, b):
    """Fraction of tokens in `a` that appear in `b`."""
    if not a or not b:
        return 0.0
    ta = set(str(a).lower().split())
    tb = set(str(b).lower().split())
    return len(ta & tb) / len(ta) if ta else 0.0


print('Utilities defined')

EXCEL_EPOCH = datetime(1899, 12, 30)

def excel_serial_to_date(val):
    """Convert Excel integer serial (e.g. 43831) or datetime → date."""
    if val is None:
        return None
    try:
        if pd.isna(val):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(val, datetime):
        return val.date()
    if isinstance(val, (int, float)) and val > 0:
        return (EXCEL_EPOCH + timedelta(days=int(val))).date()
    return None


Utilities defined


---
## Sección 1 — Parser & Normalizer

Carga los tres archivos Excel y normaliza por índice de columna (evita problemas de encoding UTF-8 en headers SAP).

In [6]:
import openpyxl, glob

print('=' * 68)
print('COLUMN PROBE — FBL1N files in Pagos directory')
print('=' * 68)

_pagos_all = sorted(
    glob.glob(os.path.join(PAGOS_DIR, '*.xlsx')) +
    glob.glob(os.path.join(PAGOS_DIR, '*.XLSX'))
)
print(f'Files found: {len(_pagos_all)}')
for _fp in _pagos_all:
    print(f'  {os.path.basename(_fp)}')

# Show headers for first 2 files (one of each format)
for _fp in _pagos_all[:2]:
    wb = openpyxl.load_workbook(_fp, read_only=True)
    ws = wb[wb.sheetnames[0]]
    hdr = list(next(ws.iter_rows(min_row=1, max_row=1, values_only=True)))
    print(f'\n{os.path.basename(_fp)}  |  {ws.max_column} cols  |  sheet: {wb.sheetnames[0]}')
    for i, h in enumerate(hdr[:25]):
        if h: print(f'  [{i:3d}] {h}')
    wb.close()

# AuxGastos probe
wb = openpyxl.load_workbook(AUXGASTOS_FILE, read_only=True)
ws = wb['Febrero_2020_KTRC']
hdr = list(next(ws.iter_rows(min_row=1, max_row=1, values_only=True)))
print(f'\nAuxGastos  ({ws.max_column} cols)')
for name, idx in list(AUXGASTOS_COLS.items())[:8]:
    print(f'  [{idx:3d}] {name:<25} -> {hdr[idx]}')
wb.close()

COLUMN PROBE — FBL1N files in Pagos directory
Files found: 11
  3400.xlsx
  3401.XLSX
  3402.XLSX
  3403.XLSX
  3406.XLSX
  3407.XLSX
  3408.XLSX
  3409.XLSX
  3413.XLSX
  3414.XLSX
  3415.XLSX

3400.xlsx  |  None cols  |  sheet: 3400 2020
  [  0] Referencia
  [  3] Cuenta
  [  4] Descripción de proveedor
  [  5] CT
  [  6] Nº doc.
  [  7] SocGLA
  [  8] Clase
  [  9] BP
  [ 10] Fecha doc.
  [ 11] Fe.contab.
  [ 12] Fecha base
  [ 13] Fecha pago
  [ 14] IO
  [ 15] Asignación
  [ 18]           Impte.MD
  [ 19] Mon.
  [ 20] Soc.
  [ 21] Cta.CP
  [ 22] Texto

3401.XLSX  |  23 cols  |  sheet: Sheet1
  [  0] Icono part.abiertas/comp.
  [  1] Acreedor
  [  2] Referencia
  [  3] Cta.contrapartida
  [  4] Descripción de proveedor
  [  5] Documento compras
  [  6] Clase de documento
  [  7] Clave contabiliz.
  [  8] Nº documento
  [  9] Fecha de documento
  [ 10] Fe.contabilización
  [ 11] Fecha de pago
  [ 12] Asignación
  [ 13] Importe en moneda doc.
  [ 14] Moneda del documento
  [ 15] Socie

In [7]:
_ktrc_re  = re.compile(r'^[A-Za-z]+_\d{4}_KTRC$')
_all_sh   = pd.read_excel(AUXGASTOS_FILE, sheet_name=None, header=0, dtype=str)

dfs_aux = []
for sh_name, df_raw in _all_sh.items():
    if not _ktrc_re.match(sh_name):
        continue
    df = df_raw.iloc[:, list(AUXGASTOS_COLS.values())].copy()
    df.columns = list(AUXGASTOS_COLS.keys())

    df['fecha_contab']  = df['fecha_contab'].apply(parse_sap_date)
    df['fecha_registro'] = df['fecha_registro'].apply(parse_sap_date)

    for col in ['sociedad', 'nro_documento', 'acreedor_id', 'clave_ct',
                'importe', 'importe_ml', 'asignacion', 'cta_contrapartida',
                'centro_costo', 'doc_compensacion']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['uuid']     = df['uuid'].apply(clean_uuid)
    df             = df[df['cuenta_gl'].astype(str).str.startswith(
                        tuple(CUENTAS_GL_GASTO_PREFIX))].copy()
    df['_anulado'] = df['anulacion'].notna() & (df['anulacion'].astype(str).str.strip() != '')
    df['_sheet']   = sh_name
    dfs_aux.append(df)

df_aux = pd.concat(dfs_aux, ignore_index=True)
df_aux['_row_id'] = df_aux.index

print(f'AuxGastos: {len(df_aux):,} expense rows from {len(dfs_aux)} sheet(s)')
print(f'  Annulled:      {df_aux["_anulado"].sum():>5,}')
print(f'  UUID present:  {df_aux["uuid"].notna().sum():>5,}  ({df_aux["uuid"].notna().mean():.1%})')
print(f'  With acreedor: {df_aux["acreedor_id"].notna().sum():>5,}  ({df_aux["acreedor_id"].notna().mean():.1%})')
df_aux.head(3)

AuxGastos: 2,404 expense rows from 1 sheet(s)
  Annulled:        388
  UUID present:    378  (15.7%)
  With acreedor:   453  (18.8%)


,sociedad,fecha_contab,ejercicio,periodo,posicion,nro_documento,clase_doc,clave_ct,cuenta_gl,descripcion_cuenta,importe,moneda,importe_ml,moneda_local,indicador_iva,folio_fiscal,uuid,cta_contrapartida,acreedor_id,nombre_proveedor,soc_gl,anulacion,fecha_registro,referencia,asignacion,texto,centro_costo,usuario,doc_compensacion,_anulado,_sheet,_row_id
0,3403,2020-02-29,2020,2020-02-01 00:00:00,2,300045533,WA,81,71101060,Suminist de Limpieza,8237.52,MXN,8237.52,MXN,NaN,NaN,NaN,11500060.0,NaN,NaN,NaN,NaN,2020-02-29,NaN,20200229.0,NaN,340312890.0,JBAUTISTA,NaN,False,Febrero_2020_KTRC,0
1,3403,2020-02-29,2020,2020-02-01 00:00:00,4,300045533,WA,81,71101060,Suminist de Limpieza,6000.00,MXN,6000.00,MXN,NaN,NaN,NaN,11500060.0,NaN,NaN,NaN,NaN,2020-02-29,NaN,20200229.0,NaN,340312890.0,JBAUTISTA,NaN,False,Febrero_2020_KTRC,1
2,3403,2020-02-29,2020,2020-02-01 00:00:00,6,300045533,WA,81,71101060,Suminist de Limpieza,3163.38,MXN,3163.38,MXN,NaN,NaN,NaN,11500060.0,NaN,NaN,NaN,NaN,2020-02-29,NaN,20200229.0,NaN,340312890.0,JBAUTISTA,NaN,False,Febrero_2020_KTRC,2


In [8]:
import glob

def load_fbl1n_file(path):
    """Load one FBL1N file — detects format by sheet name.
    Full format (123-col): sheet '3400 2020', skiprows=[1]
    Compact format (23-col): sheet 'Sheet1', no skiprows.
    """
    wb = openpyxl.load_workbook(path, read_only=True)
    sheet = wb.sheetnames[0]
    wb.close()

    if sheet != 'Sheet1':  # Full 123-column format (only 3400.xlsx)
        df_raw = pd.read_excel(path, sheet_name=sheet, header=0,
                               skiprows=[1], dtype=str)
        df = df_raw.iloc[:, list(FBL1N_COLS.values())].copy()
        df.columns = list(FBL1N_COLS.keys())
    else:                  # Compact 23-column format (all other societies)
        df_raw = pd.read_excel(path, sheet_name=sheet, header=0, dtype=str)
        # Rename by column name — handles Acreedor/Icono col-swap in 3407
        df_raw = df_raw.rename(columns=FBL1N_COLS_COMPACT)
        _keep = [c for c in FBL1N_COLS_COMPACT.values() if c in df_raw.columns]
        df = df_raw[_keep].copy()

    # ── Normalise ────────────────────────────────────────────────────────────
    for dcol in ['fecha_documento', 'fecha_contab', 'fecha_pago', 'fecha_compensacion']:
        if dcol in df.columns:
            df[dcol] = df[dcol].apply(parse_sap_date)
    df['importe'] = pd.to_numeric(df['importe'], errors='coerce').abs()
    for col in ['nro_documento', 'acreedor_id', 'sociedad', 'doc_compensacion']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['uuid'] = df['uuid'].apply(clean_uuid)
    df['_source_file'] = os.path.basename(path)
    return df


# ── Load all files from Pagos/ ────────────────────────────────────────────────
_pagos_files = sorted(
    glob.glob(os.path.join(PAGOS_DIR, '*.xlsx')) +
    glob.glob(os.path.join(PAGOS_DIR, '*.XLSX'))
)

dfs_fbl = []
for fp in tqdm(_pagos_files, desc='Loading FBL1N files'):
    _df = load_fbl1n_file(fp)
    dfs_fbl.append(_df)
    print(f'  {os.path.basename(fp):15s}  {len(_df):>7,} rows')

df_fbl1n = pd.concat(dfs_fbl, ignore_index=True)
df_fbl1n = df_fbl1n.dropna(subset=['nro_documento', 'acreedor_id'], how='all').reset_index(drop=True)

print(f'\nFBL1N combined: {len(df_fbl1n):,} rows from {len(_pagos_files)} files')
print(f'  UUID present:    {df_fbl1n["uuid"].notna().sum():>8,}  ({df_fbl1n["uuid"].notna().mean():.1%})')
print(f'  With fecha_pago: {df_fbl1n["fecha_pago"].notna().sum():>8,}')
df_fbl1n.head(3)

Loading FBL1N files:   9%|▉         | 1/11 [00:15<02:38, 15.85s/it]

  3400.xlsx         37,566 rows


Loading FBL1N files:  18%|█▊        | 2/11 [00:20<01:22,  9.22s/it]

  3401.XLSX         20,153 rows


Loading FBL1N files:  27%|██▋       | 3/11 [00:27<01:04,  8.12s/it]

...

  3413.XLSX          5,413 rows


Loading FBL1N files: 100%|██████████| 11/11 [00:49<00:00,  4.50s/it]

  3414.XLSX          2,895 rows
  3415.XLSX             74 rows



FBL1N combined: 176,414 rows from 11 files
  UUID present:     102,730  (58.2%)
  With fecha_pago:   37,516


,referencia,acreedor_id,nombre_proveedor,clave_ct,nro_documento,soc_gl,clase_doc,fecha_documento,fecha_contab,fecha_pago,importe,moneda,sociedad,cta_banco,texto,indicador_ii,via_pago,condicion_pago,usuario,uuid,fecha_compensacion,doc_compensacion,anulacion,_source_file
0,FAC-BC-CU-008111,109138.0,ADISA CABOS SAPI DE CV,31,100337546.0,NaN,RE,2002-12-07,2020-12-12,2003-01-06,517.26,MXN,3400.0,11210050,4501320310-3402- YUCA,D0,T,PO30,RMARRUFO,NaN,2020-12-13,880028605.0,NaN,3400.xlsx
1,SCD-2401,NaN,CORPORACION INMOBILIARIA KTRC SA,31,100376421.0,GL3400,ZZ,2017-11-29,2020-12-31,2017-11-29,34654.05,MXN,3400.0,19999020,SCD 2401 SERVICIOS ADMINISTRATIVOS NOVIEMBRE 2017,U3,T,NaN,VGUERRERO,027ECA4F-DB6E-5A8C-6B65-73A149A5B103,2020-12-31,880031072.0,NaN,3400.xlsx
2,FCBJTB,500004.0,FOREIGN ONE-TIME SUPPLIER,31,100166742.0,NaN,ZZ,2017-12-01,2020-04-15,2020-04-15,947.93,USD,3400.0,19999020,BOLETO DE AVION REYES SOF,D0,T,PCON,AUX_CORPO1,NaN,None,NaN,NaN,3400.xlsx


In [9]:
_raw_acr = pd.read_excel(ACREEDOR_FILE, sheet_name='Datos Maestros acreedores',
                         header=0, dtype=str)

df_acreedor = _raw_acr.iloc[:, list(ACREEDOR_COLS.values())].copy()
df_acreedor.columns = list(ACREEDOR_COLS.keys())

df_acreedor['sociedad']    = pd.to_numeric(df_acreedor['sociedad'],    errors='coerce')
df_acreedor['acreedor_id'] = pd.to_numeric(df_acreedor['acreedor_id'], errors='coerce')
df_acreedor['rfc'] = df_acreedor['rfc'].apply(
    lambda x: RFC_EXTRANJERO
    if (x is None or str(x).strip() in ('', 'nan', 'None'))
    else str(x).strip().upper()
)
df_acreedor = df_acreedor.dropna(subset=['acreedor_id']).reset_index(drop=True)

print(f'Acreedor: {len(df_acreedor):,} records  |  {df_acreedor["acreedor_id"].nunique():,} unique IDs')
df_acreedor.head(3)

Acreedor: 11,317 records  |  11,317 unique IDs


,sociedad,acreedor_id,nombre_proveedor,rfc,ramo,cuenta_asociada,condicion_pago,soc_gl
0,3401.0,100001.0,KAISENYA MARISCOS SA DE CV,KMA0806092S2,Alimentos,21021010,PO60,NaN
1,3408.0,100002.0,KAISENYA MARISCOS SA DE CV,KMA0806092S2,Alimentos,21021010,PO60,NaN
2,3406.0,100004.0,KAISENYA MARISCOS SA DE CV,KMA0806092S2,Alimentos,21021010,PO60,NaN


---
## Sección 1.5 — Ecommerce (CFDIs Recibidos)

Carga `ecommerce.xlsx` con 108,599 CFDIs vigentes. Normaliza UUID a UPPER y convierte fecha de emisión desde serial Excel.

In [ ]:
_eco_raw = pd.read_excel(
    ECOMMERCE_FILE, sheet_name=ECOMMERCE_SHEET, header=0, dtype=str,
    usecols=list(ECOMMERCE_COLS.values())
)
df_eco = _eco_raw.copy()
df_eco.columns = list(ECOMMERCE_COLS.keys())

# Filter: only Vigente records
df_eco = df_eco[df_eco['estatus'].str.strip() == 'Vigente'].copy()

# Normalize UUID to UPPER (Ecommerce stores lowercase)
df_eco['uuid'] = df_eco['uuid'].apply(clean_uuid)

# Convert Excel serial date → date
df_eco['fecha_emision'] = df_eco['fecha_emision'].apply(
    lambda x: excel_serial_to_date(pd.to_numeric(x, errors='coerce'))
)

# Numeric amounts
for col in ['subtotal', 'iva', 'total_xml', 'total_mxn', 'tipo_cambio']:
    df_eco[col] = pd.to_numeric(df_eco[col], errors='coerce')

# Normalize RFC and supplier name
df_eco['emisor_rfc']    = df_eco['emisor_rfc'].str.strip().str.upper()
df_eco['emisor_nombre'] = df_eco['emisor_nombre'].str.strip()

# Deduplicate on UUID — keep first occurrence
# (some invoices have multiple concept rows sharing one UUID)
df_eco_dedup = df_eco.dropna(subset=['uuid']).drop_duplicates(subset=['uuid'], keep='first')

print(f'Ecommerce: {len(df_eco):,} vigentes cargados')
print(f'  Con UUID:              {df_eco["uuid"].notna().sum():>8,}')
print(f'  Deduplicado por UUID:  {len(df_eco_dedup):>8,}')
print(f'  RFCs emisor únicos:    {df_eco["emisor_rfc"].nunique():>8,}')
df_eco.head(3)

---
## Sección 2 — Maestro Enricher

Enriquece AuxGastos con el RFC del proveedor. Proveedores extranjeros (sin RFC mexicano) reciben `XEXX010101000`.

In [10]:
_rfc_lkp = (
    df_acreedor.dropna(subset=['acreedor_id', 'sociedad'])
    .set_index(['sociedad', 'acreedor_id'])['rfc']
    .to_dict()
)

def _get_rfc(row):
    if pd.isna(row['acreedor_id']):
        return None
    return _rfc_lkp.get((row['sociedad'], row['acreedor_id']), RFC_EXTRANJERO)

df_aux['rfc_emisor'] = df_aux.apply(_get_rfc, axis=1)

print(f'RFC enriched:  {df_aux["rfc_emisor"].notna().sum():,}')
print(f'RFC missing:   {df_aux["rfc_emisor"].isna().sum():,}  (no acreedor_id in AuxGastos)')

RFC enriched:  453
RFC missing:   1,951  (no acreedor_id in AuxGastos)


---
## Sección 2.5 — Paso 6: AuxGastos → Ecommerce (CFDIs)

Enriquece cada línea del libro mayor con datos fiscales del CFDI correspondiente.

| Estrategia | Condición | Resultado |
|---|---|---|
| **6-A** UUID exacto | AuxGastos tiene UUID | Busca directamente en Ecommerce por UUID |
| **6-C** Difuso con RFC | Sin UUID, con acreedor_id → RFC | Filtra por RFC + fecha ±5d + similitud texto |
| **6-D** Difuso sin RFC | Sin UUID ni acreedor | Filtra por fecha ±3d + importe ±1% (alto riesgo) |

In [ ]:
# ── Paso 6-A: UUID exacto → Ecommerce ───────────────────────────────────────
_ECO_ENRICH = ['uuid', 'emisor_rfc', 'emisor_nombre', 'subtotal', 'iva',
               'total_mxn', 'concepto', 'forma_pago', 'metodo_pago', 'fecha_emision']

# Prepare Ecommerce join table with eco_ prefix
_eco_jt = df_eco_dedup[_ECO_ENRICH].rename(columns={
    'uuid':          'eco_uuid',
    'emisor_rfc':    'eco_emisor_rfc',
    'emisor_nombre': 'eco_emisor_nombre',
    'subtotal':      'eco_subtotal',
    'iva':           'eco_iva',
    'total_mxn':     'eco_total_mxn',
    'concepto':      'eco_concepto',
    'forma_pago':    'eco_forma_pago',
    'metodo_pago':   'eco_metodo_pago',
    'fecha_emision': 'eco_fecha_emision',
})

# Left join: AuxGastos.uuid → Ecommerce
_p6a = df_aux.merge(_eco_jt, left_on='uuid', right_on='eco_uuid', how='left')
_p6a['uuid_ecommerce'] = _p6a['eco_uuid']  # eco_uuid == uuid where matched
_p6a['confianza_eco']  = _p6a['eco_emisor_rfc'].apply(
    lambda x: 'EXACTO_UUID' if pd.notna(x) else None
)
_p6a_matched      = _p6a[_p6a['confianza_eco'] == 'EXACTO_UUID'].copy()
_p6a_resolved_ids = set(_p6a_matched['_row_id'])

print(f'Paso 6A — EXACTO_UUID: {len(_p6a_resolved_ids):,} registros resueltos')
print(f'  (AuxGastos con UUID: {df_aux["uuid"].notna().sum():,}')
print(f'   De ellos matcheados en Ecommerce: {len(_p6a_resolved_ids):,})')

In [ ]:
# ── Paso 6-C: Difuso con RFC → Ecommerce ─────────────────────────────────────
_p6_pending = _p6a[_p6a['confianza_eco'].isna()].copy()
_p6c_pool   = _p6_pending[_p6_pending['rfc_emisor'].notna()].copy()
_p6d_pool   = _p6_pending[_p6_pending['rfc_emisor'].isna()].copy()

print(f'Pending Paso 6: {len(_p6_pending):,}')
print(f'  Con RFC (→6C): {len(_p6c_pool):,}  |  Sin RFC (→6D): {len(_p6d_pool):,}')

# Eco join table for 6C — only relevant vendor RFCs, explicit column rename
_eco_for_6c = df_eco_dedup[
    df_eco_dedup['emisor_rfc'].isin(_p6c_pool['rfc_emisor'])
][['uuid','emisor_rfc','emisor_nombre','subtotal','iva','total_mxn',
   'concepto','forma_pago','metodo_pago','fecha_emision']].rename(columns={
    'uuid':          'uuid_ecommerce',
    'emisor_rfc':    '_eco_rfc',
    'emisor_nombre': 'eco_emisor_nombre',
    'subtotal':      'eco_subtotal',
    'iva':           'eco_iva',
    'total_mxn':     'eco_total_mxn',
    'concepto':      'eco_concepto',
    'forma_pago':    'eco_forma_pago',
    'metodo_pago':   'eco_metodo_pago',
    'fecha_emision': 'eco_fecha_emision',
}).assign(eco_emisor_rfc=lambda d: d['_eco_rfc']).copy()

df_6c = pd.DataFrame()
if len(_p6c_pool) > 0 and len(_eco_for_6c) > 0:
    # Drop eco_ cols (NaN from Paso 6A) from left side to avoid column collision
    _p6c_m = _p6c_pool.drop(
        columns=[c for c in _p6c_pool.columns
                 if c.startswith('eco_') or c in ['uuid_ecommerce','confianza_eco']],
        errors='ignore')
    _cross6c = _p6c_m.merge(
        _eco_for_6c, left_on='rfc_emisor', right_on='_eco_rfc', how='inner'
    ).drop(columns=['_eco_rfc'], errors='ignore')
    _cross6c = _cross6c.dropna(subset=['fecha_contab', 'eco_fecha_emision'])
    _cross6c['_dd'] = (
        pd.to_datetime(_cross6c['fecha_contab']) -
        pd.to_datetime(_cross6c['eco_fecha_emision'])
    ).dt.days.abs()
    _cross6c = _cross6c[_cross6c['_dd'] <= S3_FECHA_TOLERANCIA_DIAS]
    if S3_IMPORTE_TOLERANCIA_PCT is not None and len(_cross6c) > 0:
        _cross6c = _cross6c[
            _cross6c['importe_ml'].notna() & (_cross6c['importe_ml'] > 0) &
            _cross6c['eco_total_mxn'].notna()
        ]
        _cross6c['_ad'] = (
            (_cross6c['importe_ml'] - _cross6c['eco_total_mxn']).abs() /
            _cross6c['importe_ml']
        )
        _cross6c = _cross6c[_cross6c['_ad'] <= S3_IMPORTE_TOLERANCIA_PCT]
    if len(_cross6c) > 0:
        _cross6c['_ts'] = _cross6c.progress_apply(
            lambda r: text_score(r.get('descripcion_cuenta') or '',
                                 r.get('eco_concepto') or ''), axis=1
        )
        _cross6c = (_cross6c.sort_values(['_dd', '_ts'], ascending=[True, False])
                             .drop_duplicates(subset=['_row_id'], keep='first'))
        _cross6c['confianza_eco'] = np.select(
            [_cross6c['_ts'] >= TEXT_SCORE_ALTO,
             _cross6c['_ts'] >= TEXT_SCORE_MEDIO],
            ['DIFUSO_ALTO', 'DIFUSO_MEDIO'], default='DIFUSO_BAJO'
        )
        df_6c = _cross6c.copy()
_p6c_ids = set(df_6c['_row_id'].unique()) if len(df_6c) > 0 else set()
print(f'Paso 6C — Difuso con RFC: {len(_p6c_ids):,} matches')

# ── Paso 6-D: Difuso sin RFC → Ecommerce ─────────────────────────────────────
_6d_rows = []
_eco_s4 = df_eco_dedup[df_eco_dedup['fecha_emision'].notna() &
                       df_eco_dedup['total_mxn'].notna()].copy()
_eco_s4['_pd'] = pd.to_datetime(_eco_s4['fecha_emision'])

for _, aux in tqdm(_p6d_pool.iterrows(), total=len(_p6d_pool), desc='Paso 6D'):
    _d, _amt = aux['fecha_contab'], aux['importe_ml']
    if pd.isna(_d) or pd.isna(_amt) or _amt <= 0:
        continue
    _cands = _eco_s4[(_eco_s4['_pd'] - pd.Timestamp(_d)).dt.days.abs() <= S4_FECHA_TOLERANCIA_DIAS].copy()
    _cands = _cands[((_cands['total_mxn'] - _amt).abs() / _amt) <= S4_IMPORTE_TOLERANCIA_PCT]
    if len(_cands) == 0:
        continue
    _best = _cands.nsmallest(1, '_pd').iloc[0]
    row = {k: v for k, v in aux.items()
           if not (str(k).startswith('eco_') or k == 'uuid_ecommerce')}
    row.update({'uuid_ecommerce':  _best['uuid'],
                'eco_emisor_rfc':  _best['emisor_rfc'],
                'eco_emisor_nombre': _best['emisor_nombre'],
                'eco_subtotal':    _best['subtotal'],
                'eco_iva':         _best['iva'],
                'eco_total_mxn':   _best['total_mxn'],
                'eco_concepto':    _best['concepto'],
                'eco_forma_pago':  _best['forma_pago'],
                'eco_metodo_pago': _best['metodo_pago'],
                'confianza_eco':   'DIFUSO_BAJO'})
    _6d_rows.append(row)

df_6d    = pd.DataFrame(_6d_rows) if _6d_rows else pd.DataFrame()
_p6d_ids = set(df_6d['_row_id'].unique()) if len(df_6d) > 0 else set()
print(f'Paso 6D — Difuso sin RFC: {len(_p6d_ids):,} matches')

In [ ]:
# ── Merge Paso 6 enrichment back into df_aux (so Cell 13 split picks it up) ─
_ECO_OUT = ['_row_id', 'uuid_ecommerce', 'confianza_eco',
            'eco_emisor_rfc', 'eco_emisor_nombre', 'eco_subtotal', 'eco_iva',
            'eco_total_mxn', 'eco_concepto', 'eco_forma_pago', 'eco_metodo_pago']
_conf_rank_map = {'EXACTO_UUID': 0, 'DIFUSO_ALTO': 1, 'DIFUSO_MEDIO': 2, 'DIFUSO_BAJO': 3}

_p6_pieces = []
for _dfp in [_p6a_matched, df_6c, df_6d]:
    if len(_dfp) > 0:
        _cols = [c for c in _ECO_OUT if c in _dfp.columns]
        _p6_pieces.append(_dfp[_cols])

if _p6_pieces:
    _p6_all = pd.concat(_p6_pieces, ignore_index=True)
    _p6_all['_r'] = _p6_all['confianza_eco'].map(_conf_rank_map).fillna(99)
    _p6_best = (_p6_all.sort_values('_r')
                       .drop_duplicates(subset=['_row_id'], keep='first')
                       .drop(columns=['_r']))
    _existing = [c for c in _ECO_OUT if c in df_aux.columns and c != '_row_id']
    if _existing:
        df_aux = df_aux.drop(columns=_existing)
    df_aux = df_aux.merge(_p6_best, on='_row_id', how='left')

if 'confianza_eco' not in df_aux.columns:
    df_aux['confianza_eco'] = 'SIN_MATCH_ECOMMERCE'
else:
    df_aux['confianza_eco'] = df_aux['confianza_eco'].fillna('SIN_MATCH_ECOMMERCE')

_p6_noresuelto_ids = set(df_aux['_row_id']) - _p6a_resolved_ids - _p6c_ids - _p6d_ids

print('Paso 6 — Resumen')
print(f'  6A EXACTO_UUID:    {len(_p6a_resolved_ids):>5,}')
print(f'  6C Difuso con RFC: {len(_p6c_ids):>5,}')
print(f'  6D Difuso sin RFC: {len(_p6d_ids):>5,}')
print(f'  Sin match eco:     {len(_p6_noresuelto_ids):>5,}')
print()
print(df_aux['confianza_eco'].value_counts().to_string())

---
## Sección 3 — Exact Matcher

**Estrategia 1 — UUID_DIRECTO:** join directo por UUID del CFDI.

**Estrategia 2 — DOCUMENTO_CRUCE:** join por número de documento contable (BELNR).

In [11]:
# Split active vs annulled
df_anulados = df_aux[df_aux['_anulado']].copy()
df_active   = df_aux[~df_aux['_anulado']].copy()
print(f'Active: {len(df_active):,}  |  Annulled: {len(df_anulados):,}')

# FBL1N join table: rename enrichment columns with pago_ prefix to avoid collisions
_fbl1n_jt = df_fbl1n[[
    'uuid', 'nro_documento', 'acreedor_id',
    'fecha_pago', 'cta_banco', 'texto',
    'doc_compensacion', 'fecha_compensacion',
    'via_pago', 'importe', 'condicion_pago', 'fecha_contab',
]].copy().rename(columns={
    'fecha_pago':        'pago_fecha',
    'cta_banco':         'pago_cta_banco',
    'texto':             'pago_texto',
    'doc_compensacion':  'pago_doc_comp',
    'fecha_compensacion':'pago_fecha_comp',
    'via_pago':          'pago_via_pago',
    'importe':           'pago_importe',
    'condicion_pago':    'pago_condicion',
    'fecha_contab':      'pago_fecha_contab',
    'acreedor_id':       'pago_acreedor_id',
})
print('FBL1N join table prepared.')

Active: 2,016  |  Annulled: 388
FBL1N join table prepared.


In [12]:
# ─── Data Quality Diagnostic ─────────────────────────────────────────────────
print('=== DATA SCOPE ANALYSIS ===')
print('\nAuxGastos — records by sociedad:')
print(df_active['sociedad'].value_counts().to_string())
print('\nFBL1N — societies present:', df_fbl1n['sociedad'].dropna().unique())
print('\nAuxGastos active — document classes (clase_doc):')
print(df_active['clase_doc'].value_counts().to_string())
print('\nFBL1N — document classes:')
print(df_fbl1n['clase_doc'].value_counts().head(10).to_string())

_doc_overlap = len(set(df_active['nro_documento'].dropna()) & set(df_fbl1n['nro_documento'].dropna()))
_acr_overlap = len(set(df_active['acreedor_id'].dropna()) & set(df_fbl1n['acreedor_id'].dropna()))
_uuid_overlap = len(set(df_active['uuid'].dropna()) & set(df_fbl1n['uuid'].dropna()))
print(f'\nKey overlaps (AuxGastos active ↔ FBL1N):')
print(f'  nro_documento: {_doc_overlap} overlapping document numbers')
print(f'  acreedor_id:   {_acr_overlap} overlapping vendor IDs')
print(f'  uuid:          {_uuid_overlap} overlapping UUIDs')
print('\nNote: WA (material movements) and WE (goods receipts) are the majority')
print('  of AuxGastos records. These document types do NOT have direct entries')
print('  in FBL1N (vendor ledger). For these, matching relies on fuzzy search')
print('  (S4) or requires additional PO reference data.')
print('\nRE/KR/SA class documents (which DO appear in FBL1N):')
_direct_matchable = df_active[df_active['clase_doc'].isin(['RE','KR','SA','Z6','DZ','Z4','FA','FE'])]
print(f'  {len(_direct_matchable):,} records ({len(_direct_matchable)/len(df_active):.1%} of active)')

=== DATA SCOPE ANALYSIS ===

AuxGastos — records by sociedad:
sociedad
3402    578
3401    293
3407    267
3406    245
3400    242
3403    231
3409    159
3413      1

FBL1N — societies present: [3400. 3401. 3402. 3403. 3406. 3407. 3408. 3409. 3413. 3414. 3415.]

AuxGastos active — document classes (clase_doc):
clase_doc
WA    1056
WE     453
SA     312
FE     103
RE      32
Z6      26
DZ      19
Z4       3
KR       3
FA       3
CR       3
ZP       1
KZ       1
SC       1

FBL1N — document classes:
clase_doc
RE    79144
ZZ    20506
KR    16550
KA    12786
ZV     9514
KZ     8323
CZ     8046
Z6     7038
SA     4710
KG     4140

Key overlaps (AuxGastos active ↔ FBL1N):
  nro_documento: 47 overlapping document numbers
  acreedor_id:   33 overlapping vendor IDs
  uuid:          50 overlapping UUIDs

Note: WA (material movements) and WE (goods receipts) are the majority
  of AuxGastos records. These document types do NOT have direct entries
  in FBL1N (vendor ledger). For these, matching re

In [13]:
# Paso 7A — UUID match to FBL1N
# Priority: uuid_ecommerce (Paso 6 validated) > original uuid from AuxGastos
if 'uuid_ecommerce' in df_active.columns:
    df_active['_uuid_p7'] = df_active['uuid_ecommerce'].fillna(df_active['uuid'])
else:
    df_active['_uuid_p7'] = df_active['uuid']

_fbl1n_for_s1 = _fbl1n_jt[_fbl1n_jt['uuid'].notna()].rename(columns={'uuid': '_fbl_uuid'})

df_s1 = (
    df_active[df_active['_uuid_p7'].notna()]
    .merge(_fbl1n_for_s1, left_on='_uuid_p7', right_on='_fbl_uuid', how='inner')
    .drop(columns=['_fbl_uuid', '_uuid_p7'], errors='ignore')
)
# Prefer FBL1N row with pago_fecha when UUID matches multiple rows
# (same dedup logic already used in S2)
df_s1 = (
    df_s1.sort_values(['_row_id', 'pago_fecha'], na_position='last')
    .drop_duplicates(subset=['_row_id'], keep='first')
)
df_s1['confianza']  = 'EXACTO'
df_s1['estrategia'] = 'UUID_DIRECTO'

_s1_ids = set(df_s1['_row_id'].unique())
print(f'Paso 7A — UUID_DIRECTO (FBL1N): {len(df_s1):,} matches  ({len(_s1_ids):,} AuxGastos rows)')

S1 UUID_DIRECTO: 378 matches  (378 unique AuxGastos rows)


In [14]:
_s2_pool = df_active[~df_active['_row_id'].isin(_s1_ids)].copy()

df_s2 = _s2_pool.merge(
    _fbl1n_jt[_fbl1n_jt['nro_documento'].notna()],
    on='nro_documento', how='inner'
)

# Multiple FBL1N rows may share a nro_documento; prefer the one with pago_fecha
df_s2 = (
    df_s2.sort_values(['_row_id', 'pago_fecha'], na_position='last')
    .drop_duplicates(subset=['_row_id'], keep='first')
)
df_s2['confianza']  = 'EXACTO'
df_s2['estrategia'] = 'DOCUMENTO_CRUCE'

_s2_ids = set(df_s2['_row_id'].unique())
print(f'S2 DOCUMENTO_CRUCE: {len(df_s2):,} matches')

_exacto_total = len(_s1_ids) + len(_s2_ids)
print(f'\nTotal exacto: {_exacto_total:,} / {len(df_active):,}  ({_exacto_total/len(df_active):.1%})')
print(f'Pending for fuzzy: {len(df_active) - _exacto_total:,}')

S2 DOCUMENTO_CRUCE: 100 matches

Total exacto: 478 / 2,016  (23.7%)
Pending for fuzzy: 1,538


---
## Sección 4 — Fuzzy Matcher

Matching difuso para registros sin UUID ni match por documento.

| Nivel | Score de texto |
|---|---|
| `DIFUSO_ALTO` | ≥ 0.70 |
| `DIFUSO_MEDIO` | ≥ 0.40 |
| `DIFUSO_BAJO` | < 0.40 — siempre requiere revisión manual |

In [15]:
_pending    = df_active[~df_active['_row_id'].isin(_s1_ids | _s2_ids)].copy()
_s3_pool    = _pending[_pending['acreedor_id'].notna()].copy()
_s4_pool    = _pending[_pending['acreedor_id'].isna()].copy()

print(f'Pending total: {len(_pending):,}')
print(f'  With acreedor (S3): {len(_s3_pool):,}')
print(f'  No acreedor   (S4): {len(_s4_pool):,}')

# Pre-filter FBL1N to relevant acreedor_ids only
_s3_fbl = _fbl1n_jt[
    _fbl1n_jt['pago_acreedor_id'].isin(_s3_pool['acreedor_id'])
].rename(columns={'pago_acreedor_id': 'acreedor_id'}).copy()

df_s3 = pd.DataFrame()
if len(_s3_pool) > 0 and len(_s3_fbl) > 0:
    _cross = _s3_pool.merge(_s3_fbl, on='acreedor_id', how='inner')

    # Date filter
    _cross = _cross.dropna(subset=['fecha_contab', 'pago_fecha_contab'])
    _cross['_dd'] = (
        pd.to_datetime(_cross['fecha_contab']) -
        pd.to_datetime(_cross['pago_fecha_contab'])
    ).dt.days.abs()
    _cross = _cross[_cross['_dd'] <= S3_FECHA_TOLERANCIA_DIAS]

    # Amount filter — only applied when S3_IMPORTE_TOLERANCIA_PCT is not None.
    # GL line amounts (partial cost allocation) ≠ total invoice in FBL1N,
    # so a wide tolerance (e.g. 0.50) or None is recommended.
    if S3_IMPORTE_TOLERANCIA_PCT is not None:
        _cross = _cross[
            _cross['importe_ml'].notna() & (_cross['importe_ml'] > 0) &
            _cross['pago_importe'].notna()
        ]
        _cross['_ad'] = (
            (_cross['importe_ml'] - _cross['pago_importe']).abs() /
            _cross['importe_ml']
        )
        _cross = _cross[_cross['_ad'] <= S3_IMPORTE_TOLERANCIA_PCT]

    print(f'\nS3 candidates after filters: {len(_cross):,}')

    if len(_cross) > 0:
        _cross['_ts'] = _cross.progress_apply(
            lambda r: text_score(
                r.get('descripcion_cuenta') or '',
                r.get('pago_texto') or ''
            ), axis=1
        )
        _cross = (
            _cross.sort_values(['_dd', '_ts'], ascending=[True, False])
            .drop_duplicates(subset=['_row_id'], keep='first')
        )
        _cross['confianza'] = np.select(
            [_cross['_ts'] >= TEXT_SCORE_ALTO,
             _cross['_ts'] >= TEXT_SCORE_MEDIO],
            ['DIFUSO_ALTO', 'DIFUSO_MEDIO'],
            default='DIFUSO_BAJO'
        )
        _cross['estrategia'] = 'FUZZY_CON_ACREEDOR'
        df_s3 = _cross.copy()

_s3_ids = set(df_s3['_row_id'].unique()) if len(df_s3) > 0 else set()
print(f'S3 FUZZY_CON_ACREEDOR: {len(df_s3):,} matches')

Pending total: 1,538
  With acreedor (S3): 77
  No acreedor   (S4): 1,461

S3 candidates after filters: 60


100%|██████████| 60/60 [00:00<00:00, 13094.24it/s]

S3 FUZZY_CON_ACREEDOR: 23 matches


In [16]:
# Tolerances for S4 are set in the Config cell above (S4_FECHA_TOLERANCIA_DIAS, S4_S4_IMPORTE_TOLERANCIA_PCT)

_s4_final = _pending[
    ~_pending['_row_id'].isin(_s3_ids) & _pending['acreedor_id'].isna()
].copy()

_fbl1n_s4 = _fbl1n_jt[_fbl1n_jt['pago_fecha_contab'].notna() &
                       _fbl1n_jt['pago_importe'].notna()].copy()
_fbl1n_s4['_pd'] = pd.to_datetime(_fbl1n_s4['pago_fecha_contab'])

s4_rows = []
for _, aux in tqdm(_s4_final.iterrows(), total=len(_s4_final), desc='S4 fuzzy'):
    _d   = aux['fecha_contab']
    _amt = aux['importe_ml']
    if pd.isna(_d) or pd.isna(_amt) or _amt <= 0:
        continue
    _cands = _fbl1n_s4[
        (_fbl1n_s4['_pd'] - pd.Timestamp(_d)).dt.days.abs() <= S4_FECHA_TOLERANCIA_DIAS
    ].copy()
    if len(_cands) == 0:
        continue
    _cands = _cands[
        ((_cands['pago_importe'] - _amt).abs() / _amt) <= S4_IMPORTE_TOLERANCIA_PCT
    ]
    if len(_cands) == 0:
        continue
    _ref = str(aux.get('referencia') or '')
    _txt = str(aux.get('texto') or '')
    _cands['_tok'] = _cands['pago_texto'].apply(
        lambda t: max(token_overlap(_ref, str(t or '')),
                      token_overlap(_txt, str(t or '')))
    )
    _best = _cands.nlargest(1, '_tok').iloc[0]
    row = aux.to_dict()
    for k, v in _best.items():
        if k not in row:
            row[k] = v
    row['_ts'] = _best['_tok']
    s4_rows.append(row)

df_s4 = pd.DataFrame()
if s4_rows:
    df_s4 = pd.DataFrame(s4_rows)
    df_s4['confianza'] = np.select(
        [df_s4['_ts'] >= TEXT_SCORE_ALTO, df_s4['_ts'] >= TEXT_SCORE_MEDIO],
        ['DIFUSO_ALTO', 'DIFUSO_MEDIO'], default='DIFUSO_BAJO'
    )
    df_s4['estrategia'] = 'FUZZY_SIN_ACREEDOR'

_s4_ids = set(df_s4['_row_id'].unique()) if len(df_s4) > 0 else set()
print(f'S4 FUZZY_SIN_ACREEDOR: {len(df_s4):,} matches')

S4 fuzzy: 100%|██████████| 1461/1461 [00:05<00:00, 277.40it/s]

S4 FUZZY_SIN_ACREEDOR: 643 matches


In [17]:
_all_matched = _s1_ids | _s2_ids | _s3_ids | _s4_ids

df_no_res = df_active[~df_active['_row_id'].isin(_all_matched)].copy()
df_no_res['confianza']  = 'NO_RESUELTO'
df_no_res['estrategia'] = 'FALLBACK'

df_anulados_out = df_anulados.copy()
df_anulados_out['confianza']  = 'ANULADO'
df_anulados_out['estrategia'] = 'ANULADO'

print('=' * 48)
print('MATCHING SUMMARY')
print('=' * 48)
_rows = [
    ('S1 UUID_DIRECTO',       len(_s1_ids)),
    ('S2 DOCUMENTO_CRUCE',    len(_s2_ids)),
    ('S3 FUZZY_CON_ACREEDOR', len(_s3_ids)),
    ('S4 FUZZY_SIN_ACREEDOR', len(_s4_ids)),
    ('NO_RESUELTO',           len(df_no_res)),
    ('ANULADO',               len(df_anulados)),
]
for label, count in _rows:
    pct = f'({count/len(df_active):.1%})' if label != 'ANULADO' else ''
    print(f'  {label:<26} {count:>5,}  {pct}')
print('-' * 48)
_auto_pct = (len(_s1_ids)+len(_s2_ids)+len(_s3_ids)+len(_s4_ids)) / len(df_active)
_poc      = 'TARGET ALCANZADO' if _auto_pct >= 0.80 else 'BAJO TARGET (ajustar tolerancias)'
print(f'  Resolución automática: {_auto_pct:.1%}  — {_poc}')

MATCHING SUMMARY
  S1 UUID_DIRECTO              378  (18.8%)
  S2 DOCUMENTO_CRUCE           100  (5.0%)
  S3 FUZZY_CON_ACREEDOR         23  (1.1%)
  S4 FUZZY_SIN_ACREEDOR        643  (31.9%)
  NO_RESUELTO                  872  (43.3%)
  ANULADO                      388  
------------------------------------------------
  Resolución automática: 56.7%  — BAJO TARGET (ajustar tolerancias)


---
## Sección 5 — Report Builder

Consolida todos los resultados y genera `ReporteFinal_3400_202002.xlsx` con tres hojas:

- **Resultado** — todos los registros con color por nivel de confianza
- **Pendientes** — solo registros `NO_RESUELTO`
- **Métricas** — estadísticas de matching

In [18]:
_pieces = [df for df in [df_s1, df_s2, df_s3, df_s4, df_no_res, df_anulados_out]
           if len(df) > 0]
df_all = pd.concat(_pieces, ignore_index=True, sort=False)

# Final column mapping: internal name → ReporteFinal display name
_col_map = {
    'sociedad':           'Sociedad',
    'fecha_contab':       'Fe.contab.',
    'ejercicio':          'Ejercicio',
    'periodo':            'Ej./mes',
    'posicion':           'Pos.',
    'nro_documento':      'Nº doc.',
    'clase_doc':          'Clase',
    'clave_ct':           'ClvCT',
    'cuenta_gl':          'LibrMay',
    'cta_contrapartida':  'Cta.CP',
    'descripcion_cuenta': 'Txt.brv.',
    'importe':            'Importe',
    'moneda':             'Moneda',
    'importe_ml':         'Impte.ML',
    'moneda_local':       'Mon.local',
    'indicador_iva':      'Ind.imp.',
    'folio_fiscal':       'Folio Fisc',
    'uuid':               'UUID Pos.',
    'rfc_emisor':         'RFC Emisor',
    'acreedor_id':        'Acreedor',
    'nombre_proveedor':   'Nombre',
    'soc_gl':             'SocGLAsoc.',
    'anulacion':          'Anulación',
    'fecha_registro':     'Registrado',
    'referencia':         'Referencia',
    'asignacion':         'Asign.',
    'pago_texto':         'Texto',
    'pago_importe':       'Monto Pagado',
    'pago_fecha':         'Fecha de Pago',
    '_forma_pago_final':  'Forma de Pago',
    'pago_doc_comp':      'Documento contable de pago',
    'pago_cta_banco':     'Cuenta bancaria de pago',
    'centro_costo':       'Ce.coste',
    'usuario':            'Usuario',
    'eco_emisor_rfc':     'RFC Emisor (CFDI)',
    'eco_emisor_nombre':  'Emisor Nombre (CFDI)',
    'eco_subtotal':       'Subtotal CFDI',
    'eco_iva':            'IVA CFDI',
    'eco_total_mxn':      'Total MXN CFDI',
    'eco_concepto':       'Concepto CFDI',
    'eco_forma_pago':     'Forma Pago SAT',
    'eco_metodo_pago':    'Método Pago',
    'confianza_eco':      'Confianza Ecommerce',
    'confianza':          'Confianza',
    'estrategia':         'Estrategia',
}

# ── Secondary payment lookup via doc_compensacion ───────────────────────
# For rows that have pago_doc_comp but no pago_fecha, the matching
# row in FBL1N for that invoice is open; find the clearing PAYMENT
# document by looking up pago_doc_comp as a nro_documento in FBL1N.
_comp_lookup = (
    _fbl1n_jt[_fbl1n_jt['pago_fecha'].notna()][['nro_documento', 'pago_fecha', 'pago_cta_banco']]
    .drop_duplicates('nro_documento', keep='first')
    .set_index('nro_documento')
)
_need_fecha = df_all['pago_fecha'].isna() & df_all['pago_doc_comp'].notna()
if _need_fecha.sum() > 0:
    _comp_nums = pd.to_numeric(df_all.loc[_need_fecha, 'pago_doc_comp'], errors='coerce')
    _found     = _comp_nums[_comp_nums.isin(_comp_lookup.index)]
    if len(_found) > 0:
        df_all.loc[_found.index, 'pago_fecha'] = _found.map(_comp_lookup['pago_fecha'])
        _cta_null = df_all.loc[_found.index, 'pago_cta_banco'].isna()
        df_all.loc[_found.index[_cta_null], 'pago_cta_banco'] = (
            _found[_cta_null].map(_comp_lookup['pago_cta_banco'])
        )
    _filled = len(_found)
else:
    _filled = 0
print(f'Secondary doc_compensacion lookup: {_filled} additional Fecha de Pago filled')

# Coalesce 'Forma de Pago': pago_via_pago (3400.xlsx only) first,
# then eco_forma_pago (Ecommerce, when matched) as fallback.
# Compact-format FBL1N files (3401-3415) don't have via_pago.
_pago_via = df_all['pago_via_pago'] if 'pago_via_pago' in df_all.columns else pd.Series(dtype=str)
_eco_forma = df_all['eco_forma_pago'] if 'eco_forma_pago' in df_all.columns else pd.Series(dtype=str)
df_all['_forma_pago_final'] = _pago_via.fillna(_eco_forma)

_avail = [c for c in _col_map if c in df_all.columns]
df_final = (
    df_all[_avail]
    .rename(columns=_col_map)
    .sort_values(['Sociedad', 'Fe.contab.'], na_position='last')
    .reset_index(drop=True)
)

print(f'Final DataFrame: {len(df_final):,} rows × {len(df_final.columns)} columns')
df_final.head(3)

Final DataFrame: 2,404 rows × 36 columns


,Sociedad,Fe.contab.,Ejercicio,Ej./mes,Pos.,Nº doc.,Clase,ClvCT,LibrMay,Cta.CP,Txt.brv.,Importe,Moneda,Impte.ML,Mon.local,Ind.imp.,Folio Fisc,UUID Pos.,RFC Emisor,Acreedor,Nombre,SocGLAsoc.,Anulación,Registrado,Referencia,Asign.,Texto,Monto Pagado,Fecha de Pago,Forma de Pago,Documento contable de pago,Cuenta bancaria de pago,Ce.coste,Usuario,Confianza,Estrategia
0,3400,2020-02-01,2020,2020-02-01 00:00:00,3,100014790.0,RE,91,71105120,21021040.0,Mantto Licencias,0.0,USD,-10.68,MXN,D0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-02-11,120106-005E,20200201.0,Factura Booker SABE Enero 2020,400.0,2020-01-06,I,100019945.0,21021040,340099350.0,AUX_CORPO1,EXACTO,DOCUMENTO_CRUCE
1,3400,2020-02-01,2020,2020-02-01 00:00:00,1,100017747.0,SA,50,71105060,21021040.0,Otros Honorarios Pro,-7500.0,MXN,-7500.00,MXN,U3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-02-17,NaN,20190930.0,NaN,NaN,NaN,NaN,NaN,NaN,340051150.0,CMOTA,NO_RESUELTO,FALLBACK
2,3400,2020-02-01,2020,2020-02-01 00:00:00,1,100019021.0,SA,40,71106040,21021040.0,Comis Agen de Viajes,184.0,USD,3456.49,MXN,U3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-02-24,RCI-52962,20191231.0,NaN,NaN,NaN,NaN,NaN,NaN,340099350.0,CMOTA,NO_RESUELTO,FALLBACK


In [19]:
writer = pd.ExcelWriter(OUTPUT_FILE, engine='xlsxwriter',
                        engine_kwargs={'options': {'nan_inf_to_errors': True}})
wb_out = writer.book

_hdr_fmt = wb_out.add_format({
    'bold': True, 'bg_color': '1F3864', 'font_color': 'FFFFFF',
    'border': 1, 'align': 'center', 'valign': 'vcenter', 'text_wrap': True,
})
_row_fmts = {k: wb_out.add_format({'bg_color': v, 'border': 1})
             for k, v in OUTPUT_COLORS.items()}
_plain    = wb_out.add_format({'border': 1})


def _clean(v):
    """Replace NaN/Inf/date with Excel-safe types."""
    if v is None:
        return ''
    try:
        if isinstance(v, float) and (np.isnan(v) or np.isinf(v)):
            return ''
    except (TypeError, ValueError):
        pass
    import datetime as _dt
    if isinstance(v, (_dt.date, _dt.datetime)):
        return str(v)
    return v


def write_colored(df, sheet_name):
    ws = wb_out.add_worksheet(sheet_name)
    writer.sheets[sheet_name] = ws
    for ci, col in enumerate(df.columns):
        ws.write(0, ci, col, _hdr_fmt)
        ws.set_column(ci, ci, min(max(len(str(col)) + 2, 10), 30))
    _ci_conf = df.columns.tolist().index('Confianza') if 'Confianza' in df.columns else None
    for ri, row in enumerate(df.itertuples(index=False), start=1):
        conf = str(row[_ci_conf]) if _ci_conf is not None else ''
        fmt  = _row_fmts.get(conf, _plain)
        ws.write_row(ri, 0, [_clean(v) for v in row], fmt)
    ws.freeze_panes(1, 0)
    ws.autofilter(0, 0, len(df), len(df.columns) - 1)


# Sheet: Resultado
write_colored(df_final, 'Resultado')
print(f'Sheet Resultado: {len(df_final):,} rows')

# Sheet: Pendientes
df_pend = df_final[df_final['Confianza'] == 'NO_RESUELTO'].copy()
write_colored(df_pend, 'Pendientes')
print(f'Sheet Pendientes: {len(df_pend):,} rows')

# Sheet: Métricas
_metrics = {
    'total_registros':    len(df_active) + len(df_anulados),
    'activos':            len(df_active),
    'con_uuid_origen':    int(df_aux['uuid'].notna().sum()),
    'match_exacto_uuid':  len(_s1_ids),
    'match_exacto_doc':   len(_s2_ids),
    'match_difuso_alto':  int((df_all.get('confianza', pd.Series()) == 'DIFUSO_ALTO').sum()),
    'match_difuso_medio': int((df_all.get('confianza', pd.Series()) == 'DIFUSO_MEDIO').sum()),
    'match_difuso_bajo':  int((df_all.get('confianza', pd.Series()) == 'DIFUSO_BAJO').sum()),
    'no_resuelto':        len(df_no_res),
    'anulado':            len(df_anulados),
    'pct_resolucion':     round(_auto_pct, 4),
}
df_met = pd.DataFrame(list(_metrics.items()), columns=['Métrica', 'Valor'])
df_met.to_excel(writer, sheet_name='Métricas', index=False)
_wsm = writer.sheets['Métricas']
_wsm.set_column(0, 0, 28)
_wsm.set_column(1, 1, 14)
print(f'Sheet Métricas: {len(df_met)} metrics')

writer.close()
print(f'\nSaved: {OUTPUT_FILE}')

Sheet Resultado: 2,404 rows
Sheet Pendientes: 872 rows
Sheet Métricas: 11 metrics

Saved: output/ReporteFinal_3400_202002.xlsx


---
## Sección 6 — Métricas de Éxito

**Target mínimo PoC:** ≥ 80% resolución automática.

In [ ]:
print('=' * 52)
print('MÉTRICAS FINALES')
print('=' * 52)
for k, v in _metrics.items():
    display_v = f'{v:.1%}' if k == 'pct_resolucion' else f'{v:,}'
    print(f'  {k:<28}  {display_v:>10}')
print('-' * 52)
_status = 'TARGET ALCANZADO ✓' if _metrics['pct_resolucion'] >= 0.80 else 'BAJO TARGET ✗'
print(f'  PoC Status: {_status}')
print('=' * 52)
print(f'\nArchivo generado: {OUTPUT_FILE}')

print(df_met.to_string(index=False))

MÉTRICAS FINALES
  total_registros                    2,404
  activos                            2,016
  con_uuid_origen                      378
  match_exacto_uuid                    378
  match_exacto_doc                     100
  match_difuso_alto                     93
  match_difuso_medio                     8
  match_difuso_bajo                    565
  no_resuelto                          872
  anulado                              388
  pct_resolucion                     56.8%
----------------------------------------------------
  PoC Status: BAJO TARGET ✗

Archivo generado: output/ReporteFinal_3400_202002.xlsx


AttributeError: The '.style' accessor requires jinja2